# YOLO26 Video Inference

Runs a YOLO26 `.pt` model on every frame of an MP4 that is already captured at 1 fps.

**Upload your `.pt` model and `.mp4` video when prompted.**

## Step 1: Install & import

In [ ]:
!pip install -q ultralytics

import cv2
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from google.colab import files

## Step 2: Upload model and video

In [ ]:
print('Upload your YOLO26 .pt model file:')
model_upload = files.upload()
model_path = next(k for k in model_upload if k.endswith('.pt'))

print('\nUpload your .mp4 video (already at 1 fps):')
video_upload = files.upload()
video_path = next(k for k in video_upload if k.lower().endswith('.mp4'))

print(f'\nModel: {model_path}')
print(f'Video: {video_path}')

## Step 3: Run tracking on every frame

Since the video is already at 1 fps, we process every frame (no sub-sampling).
Uses `model.track(..., persist=True)` with ByteTrack so each object keeps a
stable `track_id` across frames (masks too, if this is a `-seg` model).
Annotated frames are written to `output.mp4` and per-detection rows to `detections.csv`.

In [ ]:
import numpy as np
import torch

model = YOLO(model_path)

# --- Tunables ----------------------------------------------------------------
# Per-class confidence thresholds. Classes not listed use DEFAULT_CONF.
# Keys are matched case-insensitively against model.names.
CLASS_CONF = {
    'bench': 0.15,
    'rough trail': 0.15,
}
DEFAULT_CONF = 0.35

# Overlap suppression: drop `weaker` whenever it overlaps `stronger`.
SUPPRESS_RULES = [
    ('crushed gravel trail', 'rough trail'),  # rough trail always wins
]
IOU_SUPPRESS = 0.3  # mask-IoU; lower toward 0 for looser "any overlap" behavior

# FP16 inference if a CUDA GPU is available (ignored on CPU).
USE_HALF = torch.cuda.is_available()
# -----------------------------------------------------------------------------

print('Model class names:', model.names)
print(f'FP16 inference: {USE_HALF}')
name_to_id = {v.lower(): k for k, v in model.names.items()}

def conf_for(class_id):
    return CLASS_CONF.get(model.names.get(class_id, '').lower(), DEFAULT_CONF)

def iou_mask(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union > 0 else 0.0

GLOBAL_CONF = min([DEFAULT_CONF] + list(CLASS_CONF.values()))

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 1.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {width}x{height}, {fps:.2f} fps, {total} frames')

out_video = 'output.mp4'
writer = cv2.VideoWriter(out_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

rows = []
frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break

    results = model.track(frame, persist=True, tracker='bytetrack.yaml',
                          conf=GLOBAL_CONF, half=USE_HALF, verbose=False)[0]

    if results.boxes is not None and len(results.boxes) > 0:
        confs = results.boxes.conf.cpu().numpy()
        clss = results.boxes.cls.cpu().numpy().astype(int)
        masks_np = (results.masks.data.cpu().numpy().astype(bool)
                    if results.masks is not None else None)

        # 1. Per-class confidence filter
        keep = [i for i, (c, k) in enumerate(zip(confs, clss)) if c >= conf_for(int(k))]

        # 2. Overlap suppression using mask IoU (drop weaker when overlapping stronger)
        suppressed = set()
        if masks_np is not None:
            for weaker_name, stronger_name in SUPPRESS_RULES:
                w_id = name_to_id.get(weaker_name.lower())
                s_id = name_to_id.get(stronger_name.lower())
                if w_id is None or s_id is None:
                    continue
                for i in keep:
                    if int(clss[i]) != w_id:
                        continue
                    for j in keep:
                        if int(clss[j]) != s_id:
                            continue
                        if iou_mask(masks_np[i], masks_np[j]) > IOU_SUPPRESS:
                            suppressed.add(i)
                            break
        keep = [i for i in keep if i not in suppressed]

        # Filter Results in place so plot() and CSV both see only survivors
        mask_t = torch.zeros(len(results.boxes), dtype=torch.bool,
                             device=results.boxes.data.device)
        if keep:
            mask_t[keep] = True
        results.boxes = results.boxes[mask_t]
        if results.masks is not None:
            results.masks = results.masks[mask_t]

    annotated = results.plot()
    writer.write(annotated)

    if results.boxes is not None and len(results.boxes) > 0:
        boxes = results.boxes.xyxy.cpu().numpy()
        confs = results.boxes.conf.cpu().numpy()
        clss = results.boxes.cls.cpu().numpy().astype(int)
        ids = (results.boxes.id.cpu().numpy().astype(int)
               if results.boxes.id is not None
               else [-1] * len(boxes))
        for (x1, y1, x2, y2), conf, cls, tid in zip(boxes, confs, clss, ids):
            rows.append({
                'frame': frame_idx,
                't_sec': frame_idx,
                'track_id': int(tid),
                'class_id': int(cls),
                'class_name': model.names.get(int(cls), str(cls)),
                'conf': float(conf),
                'x1': float(x1), 'y1': float(y1),
                'x2': float(x2), 'y2': float(y2),
            })

    frame_idx += 1
    if frame_idx % 25 == 0:
        print(f'  processed {frame_idx}/{total} frames')

cap.release()
writer.release()

det = pd.DataFrame(rows)
det.to_csv('detections.csv', index=False)
n_tracks = det['track_id'].nunique() if len(det) else 0
print(f'\nDone. {frame_idx} frames processed, {len(det)} detections, {n_tracks} unique tracks.')
det.head()

## Step 4: Download results

In [ ]:
files.download('output.mp4')
files.download('detections.csv')